# Relevant Python 3.11 Changes — Advanced Tutorial Problems with Solutions

Python 3.11 contains several changes that become especially valuable in larger programs: better diagnostics, structured concurrency, grouped failures, richer typing, a TOML parser, safer integer conversion, and new regular-expression controls.

This notebook uses a **tutorial style**. We will not jump straight to final implementations. Each problem is divided into small steps, short experiments, a complete solution, tests, and a list of practical lessons.

The problems are intentionally different from a simple feature tour. They are based on realistic boundaries such as configuration loading, batch validation, concurrent services, protocol parsing, and typed APIs.

All examples are self-contained. They require Python 3.11 or newer and no third-party packages.

For the complete release details, see [What’s New in Python 3.11](https://docs.python.org/3.11/whatsnew/3.11.html). Additional primary references are linked inside the relevant sections.

## How to use this notebook

Read the Markdown cells in order. Run the small exploratory cells before the solution cells. The tests are deliberately close to the code they exercise so that every design decision remains visible.

In [1]:
import sys

assert sys.version_info >= (3, 11), "Python 3.11 or newer is required."
print(sys.version)

3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


We will use a tiny exception assertion helper instead of introducing a testing framework.

In [2]:
def assert_raises(expected_exception, function, /, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except expected_exception as ex:
        return ex
    except Exception as ex:
        raise AssertionError(
            f"Expected {expected_exception.__name__}, got {type(ex).__name__}"
        ) from ex
    raise AssertionError(f"Expected {expected_exception.__name__} to be raised")

## Roadmap

1. Precise source locations for formula diagnostics.  
2. Exception notes across layered code.  
3. Nested `ExceptionGroup` validation.  
4. Fail-fast and tolerant `TaskGroup` designs.  
5. Local and global `asyncio.timeout()` budgets.  
6. Typed TOML configuration loading.  
7. A `StrEnum` state machine.  
8. A fluent API using `typing.Self`.  
9. Partial payloads with `Required` and `NotRequired`.  
10. Shape-aware generics with `TypeVarTuple`.  
11. A `LiteralString` query boundary.  
12. Atomic regex groups and possessive quantifiers.  
13. Application limits for large decimal identifiers.  
14. A combined concurrent report-runner capstone.

---

## Problem 1 — Build a fine-grained formula diagnostic

Python 3.11 implements [PEP 657](https://peps.python.org/pep-0657/), which records more precise source positions for expressions. Standard tracebacks can now point at the part of a line that actually failed.

### Scenario

A reporting application evaluates trusted analyst formulas. When a formula fails, we want to show the exception and underline the relevant expression fragment.

### Step 1 — Define a presentation-neutral result

The diagnostic object should contain structured data. Rendering is a separate concern.

In [3]:
from dataclasses import dataclass


@dataclass(frozen=True, slots=True)
class FormulaDiagnostic:
    exception_type: str
    message: str
    source: str
    start_column: int
    end_column: int

    @property
    def fragment(self) -> str:
        return self.source[self.start_column:self.end_column]

    def render(self) -> str:
        width = max(1, self.end_column - self.start_column)
        pointer = " " * self.start_column + "^" * width
        return (
            f"{self.exception_type}: {self.message}\n"
            f"{self.source}\n"
            f"{pointer}"
        )

### Step 2 — Distinguish compilation from evaluation

`compile()` raises `SyntaxError`. A valid expression can later raise an ordinary runtime exception during `eval()`. Position data is obtained differently in the two cases.

### Step 3 — Implement the diagnostic boundary

The restricted built-ins in this teaching example reduce accidental access, but they do **not** make `eval()` a safe sandbox for untrusted input.

In [4]:
import traceback
from collections.abc import Mapping
from typing import Any


def evaluate_formula(expression: str, variables: Mapping[str, Any]):
    try:
        compiled = compile(expression, "<formula>", "eval")
        return eval(compiled, {"__builtins__": {}}, dict(variables))
    except SyntaxError as ex:
        start = max(0, (ex.offset or 1) - 1)
        end = max(start + 1, (ex.end_offset or ex.offset or 1) - 1)
        return FormulaDiagnostic(
            type(ex).__name__, ex.msg, expression, start, end
        )
    except Exception as ex:
        frames = traceback.extract_tb(ex.__traceback__)
        frame = next(
            (item for item in reversed(frames) if item.filename == "<formula>"),
            frames[-1],
        )
        start = frame.colno or 0
        end = frame.end_colno or max(start + 1, len(expression))
        return FormulaDiagnostic(
            type(ex).__name__, str(ex), expression, start, end
        )

### Step 4 — Test a successful formula

In [5]:
context = {
    "subtotal": 250.0,
    "discounts": {"vip": 0.20},
    "tier": "vip",
}

assert evaluate_formula(
    "subtotal * (1 - discounts[tier])",
    context,
) == 200.0

### Step 5 — Test a runtime failure

In [6]:
runtime_diagnostic = evaluate_formula(
    "subtotal / discounts[tier]",
    {"subtotal": 250.0, "discounts": {"vip": 0.0}, "tier": "vip"},
)

assert isinstance(runtime_diagnostic, FormulaDiagnostic)
assert runtime_diagnostic.exception_type == "ZeroDivisionError"
print(runtime_diagnostic.render())

ZeroDivisionError: float division by zero
subtotal / discounts[tier]
^^^^^^^^^^^^^^^^^^^^^^^^^^


### Step 6 — Test a syntax failure

In [7]:
syntax_diagnostic = evaluate_formula(
    "subtotal * (1 - discounts[tier]",
    context,
)

assert isinstance(syntax_diagnostic, FormulaDiagnostic)
assert syntax_diagnostic.exception_type == "SyntaxError"
print(syntax_diagnostic.render())

SyntaxError: '(' was never closed
subtotal * (1 - discounts[tier]
           ^


### Lessons

- Keep source-position collection separate from rendering.
- Preserve the original exception type and message.
- Use a dedicated expression grammar or AST allow-list for untrusted expressions.
- Fine-grained locations improve diagnostics; they do not replace tracebacks or logging.

---

## Problem 2 — Enrich an exception as it crosses application layers

[PEP 678](https://peps.python.org/pep-0678/) added `BaseException.add_note()`. Notes preserve the original exception while appending context discovered by higher-level code.

### Scenario

A subscription importer converts raw text, knows the source file and row, and finally knows the batch ID. Each layer should contribute only the context it owns.

### Step 1 — Define the domain object and focused parser

In [8]:
from datetime import date


@dataclass(frozen=True, slots=True)
class Subscription:
    customer_id: int
    starts_on: date
    monthly_price: float


def parse_subscription(row: dict[str, str]) -> Subscription:
    return Subscription(
        customer_id=int(row["customer_id"]),
        starts_on=date.fromisoformat(row["starts_on"]),
        monthly_price=float(row["monthly_price"]),
    )

The low-level parser should not know file names or batch identifiers. That separation keeps it reusable.

### Step 2 — Add source context and re-raise the same exception

In [9]:
def parse_subscription_row(
    row: dict[str, str],
    *,
    filename: str,
    row_number: int,
) -> Subscription:
    try:
        return parse_subscription(row)
    except (KeyError, ValueError) as ex:
        ex.add_note(f"source={filename!r}, row={row_number}")
        raise

### Step 3 — Add batch context at the orchestration layer

In [10]:
def import_subscription(
    row: dict[str, str],
    *,
    filename: str,
    row_number: int,
    batch_id: str,
) -> Subscription:
    try:
        return parse_subscription_row(
            row,
            filename=filename,
            row_number=row_number,
        )
    except (KeyError, ValueError) as ex:
        ex.add_note(f"batch_id={batch_id!r}")
        raise

### Step 4 — Inspect the accumulated notes

In [11]:
bad_subscription = {
    "customer_id": "1042",
    "starts_on": "2026-02-30",
    "monthly_price": "19.95",
}

try:
    import_subscription(
        bad_subscription,
        filename="subscriptions-07.csv",
        row_number=18,
        batch_id="nightly-2026-07-30",
    )
except ValueError as ex:
    noted_error = ex

assert noted_error.__notes__ == [
    "source='subscriptions-07.csv', row=18",
    "batch_id='nightly-2026-07-30'",
]
noted_error.__notes__

["source='subscriptions-07.csv', row=18", "batch_id='nightly-2026-07-30'"]

### Why not overwrite the message?

The original invalid-date message is still valuable. Notes add operational context without hiding that technical detail.

### Lessons

- Add context where it becomes available.
- Use exception chaining when the abstraction changes; use notes when the exception type remains appropriate.
- Keep notes concise and avoid duplicate notes during retries.
- Catch only the exception types the layer understands.

---

## Problem 3 — Report every independent batch validation failure

[PEP 654](https://peps.python.org/pep-0654/) introduced `ExceptionGroup`, `BaseExceptionGroup`, and `except*`. These features allow several unrelated failures to travel together while preserving nested structure.

### Scenario

A product catalog import should validate every field in every record. Users should receive one complete report rather than fixing one error per run.

### Step 1 — Define exception categories that callers can select

In [12]:
class ProductValidationError(ValueError):
    pass


class MissingProductField(ProductValidationError):
    pass


class InvalidProductValue(ProductValidationError):
    pass

### Step 2 — Validate one product and collect independent field errors

In [13]:
def validate_product(record: dict[str, object], *, index: int) -> None:
    errors: list[Exception] = []

    sku = record.get("sku")
    if not isinstance(sku, str) or not sku.strip():
        errors.append(MissingProductField("sku must be a non-empty string"))

    price = record.get("price")
    if not isinstance(price, (int, float)) or isinstance(price, bool) or price <= 0:
        errors.append(InvalidProductValue("price must be a positive number"))

    stock = record.get("stock")
    if not isinstance(stock, int) or isinstance(stock, bool) or not 0 <= stock <= 10_000:
        errors.append(InvalidProductValue("stock must be an integer from 0 to 10,000"))

    if errors:
        for error in errors:
            error.add_note(f"record_index={index}")
        raise ExceptionGroup(f"invalid product at index {index}", errors)

### Step 3 — Preserve one nested group per record

In [14]:
def validate_catalog(records: list[dict[str, object]]) -> None:
    record_groups: list[Exception] = []

    for index, record in enumerate(records):
        try:
            validate_product(record, index=index)
        except ExceptionGroup as group:
            record_groups.append(group)

    if record_groups:
        raise ExceptionGroup("catalog validation failed", record_groups)

### Step 4 — Prepare a catalog with several failures

In [15]:
catalog = [
    {"sku": "A-100", "price": 12.50, "stock": 40},
    {"sku": "", "price": -2, "stock": 40},
    {"sku": "C-300", "price": 8.25, "stock": 50_000},
    {"price": 4.00, "stock": "many"},
]

We need a small helper that flattens a group only for assertions and display. The application itself keeps the nested structure.

In [16]:
def exception_leaves(group: BaseExceptionGroup) -> list[BaseException]:
    leaves: list[BaseException] = []
    for item in group.exceptions:
        if isinstance(item, BaseExceptionGroup):
            leaves.extend(exception_leaves(item))
        else:
            leaves.append(item)
    return leaves

### Step 5 — Handle different leaf types with `except*`

In [17]:
missing_messages: list[str] = []
invalid_messages: list[str] = []

try:
    validate_catalog(catalog)
except* MissingProductField as group:
    missing_messages.extend(str(error) for error in exception_leaves(group))
except* InvalidProductValue as group:
    invalid_messages.extend(str(error) for error in exception_leaves(group))

assert len(missing_messages) == 2
assert len(invalid_messages) == 3

Every matching `except*` clause may run on its own subgroup. This differs from ordinary `except`, where the first matching clause wins.

### Step 6 — Split a group dynamically

In [18]:
try:
    validate_catalog(catalog)
except ExceptionGroup as group:
    value_group, remaining_group = group.split(InvalidProductValue)

assert value_group is not None
assert remaining_group is not None
assert len(exception_leaves(value_group)) == 3
assert len(exception_leaves(remaining_group)) == 2

### Lessons

- Group only independent failures.
- Preserve nesting when it carries useful structure.
- Use domain-specific subclasses so callers can select subgroups.
- Flatten groups at presentation boundaries, not during collection.
- Remember that `except*` handles matching leaves in parallel subgroups.

---

## Problem 4 — Choose fail-fast or tolerant structured concurrency

`asyncio.TaskGroup`, new in Python 3.11, gives related tasks a structured lifetime. The asynchronous context does not finish until its child tasks finish or are cancelled.

### Scenario

A monitoring service polls several sensors concurrently. An atomic snapshot should fail fast, while a status dashboard may accept partial results.

### Step 1 — Build a deterministic sensor simulator

In [19]:
import asyncio


class SensorError(RuntimeError):
    pass


@dataclass(frozen=True, slots=True)
class SensorSpec:
    name: str
    delay: float
    value: float | None = None
    failure: str | None = None


async def poll_sensor(spec: SensorSpec) -> float:
    await asyncio.sleep(spec.delay)
    if spec.failure is not None:
        raise SensorError(f"{spec.name}: {spec.failure}")
    assert spec.value is not None
    return spec.value

### Step 2 — Implement the fail-fast policy

If one sensor failure invalidates the complete snapshot, let the child exception escape. `TaskGroup` will cancel unfinished siblings and raise a group.

In [20]:
async def poll_snapshot_fail_fast(specs: list[SensorSpec]) -> dict[str, float]:
    tasks: dict[str, asyncio.Task[float]] = {}

    async with asyncio.TaskGroup() as group:
        for spec in specs:
            tasks[spec.name] = group.create_task(
                poll_sensor(spec),
                name=f"sensor:{spec.name}",
            )

    return {name: task.result() for name, task in tasks.items()}

### Step 3 — Observe fail-fast behavior

In [21]:
sensor_specs = [
    SensorSpec("temperature", 0.01, value=21.5),
    SensorSpec("humidity", 0.02, failure="checksum mismatch"),
    SensorSpec("pressure", 0.05, value=101.2),
]

sensor_errors: list[str] = []

try:
    await poll_snapshot_fail_fast(sensor_specs)
except* SensorError as group:
    sensor_errors.extend(str(error) for error in exception_leaves(group))

assert sensor_errors == ["humidity: checksum mismatch"]

The pressure task may be cancelled because its result would not create a valid atomic snapshot.

### Step 4 — Convert expected failures into result values

A tolerant dashboard considers an unavailable sensor part of the normal result model.

In [22]:
@dataclass(frozen=True, slots=True)
class SensorResult:
    name: str
    value: float | None
    error: str | None


async def poll_sensor_safely(spec: SensorSpec) -> SensorResult:
    try:
        value = await poll_sensor(spec)
    except SensorError as ex:
        return SensorResult(spec.name, None, str(ex))
    return SensorResult(spec.name, value, None)

### Step 5 — Implement the tolerant policy

In [23]:
async def poll_snapshot_tolerant(specs: list[SensorSpec]) -> list[SensorResult]:
    tasks: list[asyncio.Task[SensorResult]] = []

    async with asyncio.TaskGroup() as group:
        for spec in specs:
            tasks.append(group.create_task(poll_sensor_safely(spec)))

    return [task.result() for task in tasks]

### Step 6 — Verify that all sensors are represented

In [24]:
tolerant_results = await poll_snapshot_tolerant(sensor_specs)

assert [result.name for result in tolerant_results] == [
    "temperature", "humidity", "pressure"
]
assert tolerant_results[1].error == "humidity: checksum mismatch"
assert tolerant_results[2].value == 101.2

tolerant_results

[SensorResult(name='temperature', value=21.5, error=None),
 SensorResult(name='humidity', value=None, error='humidity: checksum mismatch'),
 SensorResult(name='pressure', value=101.2, error=None)]

### Lessons

- Decide whether a child failure invalidates the parent operation.
- Let fatal failures escape; convert expected partial failures to values.
- Name tasks for diagnostics.
- Keep task ownership inside the group.
- Do not swallow `CancelledError` after cleanup.

---

## Problem 5 — Apply local and global asynchronous deadlines

Python 3.11 added `asyncio.timeout()`, an asynchronous context manager for deadlines.

### Scenario

A dashboard renders three widgets. Each widget has a local budget, while the complete dashboard has a separate overall budget.

### Step 1 — Define the simulated work and result model

In [25]:
@dataclass(frozen=True, slots=True)
class WidgetSpec:
    name: str
    render_time: float
    budget: float


@dataclass(frozen=True, slots=True)
class WidgetResult:
    name: str
    html: str | None
    timed_out: bool


async def render_widget(spec: WidgetSpec) -> str:
    await asyncio.sleep(spec.render_time)
    return f"<{spec.name}>"

### Step 2 — Treat a local timeout as an expected widget result

In [26]:
async def render_widget_with_budget(spec: WidgetSpec) -> WidgetResult:
    try:
        async with asyncio.timeout(spec.budget):
            html = await render_widget(spec)
    except TimeoutError:
        return WidgetResult(spec.name, None, True)
    return WidgetResult(spec.name, html, False)

### Step 3 — Protect the whole operation with an outer timeout

In [27]:
async def render_dashboard(
    specs: list[WidgetSpec],
    *,
    overall_budget: float,
) -> list[WidgetResult]:
    tasks: list[asyncio.Task[WidgetResult]] = []

    async with asyncio.timeout(overall_budget):
        async with asyncio.TaskGroup() as group:
            for spec in specs:
                tasks.append(group.create_task(render_widget_with_budget(spec)))

    return [task.result() for task in tasks]

### Step 4 — Verify a local timeout

In [28]:
widgets = [
    WidgetSpec("sales", 0.01, 0.05),
    WidgetSpec("inventory", 0.08, 0.02),
    WidgetSpec("alerts", 0.015, 0.05),
]

widget_results = await render_dashboard(widgets, overall_budget=0.20)
assert [item.timed_out for item in widget_results] == [False, True, False]
widget_results

[WidgetResult(name='sales', html='<sales>', timed_out=False),
 WidgetResult(name='inventory', html=None, timed_out=True),
 WidgetResult(name='alerts', html='<alerts>', timed_out=False)]

### Step 5 — Verify the global timeout

In [29]:
try:
    await render_dashboard(
        [WidgetSpec("one", 0.08, 0.20), WidgetSpec("two", 0.08, 0.20)],
        overall_budget=0.01,
    )
except TimeoutError:
    global_timeout_triggered = True
else:
    global_timeout_triggered = False

assert global_timeout_triggered

### Lessons

- Put deadlines around meaningful operations.
- Distinguish per-item timeouts from parent-operation timeouts.
- Let the event loop manage monotonic deadlines.
- Make resource cleanup cancellation-safe.

---

## Problem 6 — Parse and validate a TOML deployment file

Python 3.11 added [`tomllib`](https://docs.python.org/3.11/library/tomllib.html), a standard-library TOML 1.0 parser. It reads TOML but does not write it.

### Scenario

A deployment file describes the application, environment, replica count, resources, and feature flags. Parsing must be followed by complete schema validation.

### Step 1 — Define the TOML input

In [30]:
DEPLOYMENT_TOML = r"""
[application]
name = "billing-api"
environment = "production"
replicas = 3

[application.resources]
cpu = 1.5
memory_mb = 1024

[features]
audit_log = true
experimental_cache = false
"""

### Step 2 — Parse floating-point values as exact decimals

In [31]:
import tomllib
from decimal import Decimal


raw_deployment = tomllib.loads(DEPLOYMENT_TOML, parse_float=Decimal)
assert raw_deployment["application"]["resources"]["cpu"] == Decimal("1.5")

### Step 3 — Define external values and domain models

In [32]:
from enum import StrEnum


class Environment(StrEnum):
    DEVELOPMENT = "development"
    STAGING = "staging"
    PRODUCTION = "production"


@dataclass(frozen=True, slots=True)
class ResourceLimits:
    cpu: Decimal
    memory_mb: int


@dataclass(frozen=True, slots=True)
class DeploymentConfig:
    name: str
    environment: Environment
    replicas: int
    resources: ResourceLimits
    features: dict[str, bool]

### Step 4 — Validate every independent field before raising

In [33]:
def deployment_from_mapping(data: dict[str, object]) -> DeploymentConfig:
    errors: list[Exception] = []

    application = data.get("application")
    features = data.get("features")

    if not isinstance(application, dict):
        errors.append(ValueError("[application] table is required"))
        application = {}
    if not isinstance(features, dict):
        errors.append(ValueError("[features] table is required"))
        features = {}

    name = application.get("name")
    if not isinstance(name, str) or not name.strip():
        errors.append(ValueError("application.name must be a non-empty string"))

    try:
        environment = Environment(application.get("environment"))
    except (TypeError, ValueError) as ex:
        ex.add_note("field=application.environment")
        errors.append(ex)
        environment = Environment.DEVELOPMENT

    replicas = application.get("replicas")
    if not isinstance(replicas, int) or isinstance(replicas, bool) or not 1 <= replicas <= 100:
        errors.append(ValueError("application.replicas must be an integer from 1 to 100"))

    resources = application.get("resources")
    if not isinstance(resources, dict):
        errors.append(ValueError("[application.resources] table is required"))
        resources = {}

    cpu = resources.get("cpu")
    if not isinstance(cpu, (int, Decimal)) or isinstance(cpu, bool) or cpu <= 0:
        errors.append(ValueError("application.resources.cpu must be positive"))

    memory_mb = resources.get("memory_mb")
    if not isinstance(memory_mb, int) or isinstance(memory_mb, bool) or memory_mb < 128:
        errors.append(ValueError("application.resources.memory_mb must be at least 128"))

    invalid_flags = {
        key: value
        for key, value in features.items()
        if not isinstance(key, str) or not isinstance(value, bool)
    }
    if invalid_flags:
        errors.append(ValueError(f"feature flags must be string-to-bool pairs: {invalid_flags!r}"))

    if errors:
        raise ExceptionGroup("invalid deployment configuration", errors)

    return DeploymentConfig(
        name=name,
        environment=environment,
        replicas=replicas,
        resources=ResourceLimits(Decimal(cpu), memory_mb),
        features=dict(features),
    )

### Step 5 — Add the configuration source as a note

In [34]:
def load_deployment_config(text: str, *, source_name: str) -> DeploymentConfig:
    try:
        parsed = tomllib.loads(text, parse_float=Decimal)
        return deployment_from_mapping(parsed)
    except (tomllib.TOMLDecodeError, ExceptionGroup) as ex:
        ex.add_note(f"configuration_source={source_name!r}")
        raise

### Step 6 — Test the valid document

In [35]:
deployment = load_deployment_config(
    DEPLOYMENT_TOML,
    source_name="deployment.toml",
)

assert deployment.environment is Environment.PRODUCTION
assert deployment.resources.cpu == Decimal("1.5")
assert deployment.features["audit_log"] is True
deployment

DeploymentConfig(name='billing-api', environment=<Environment.PRODUCTION: 'production'>, replicas=3, resources=ResourceLimits(cpu=Decimal('1.5'), memory_mb=1024), features={'audit_log': True, 'experimental_cache': False})

### Step 7 — Test several invalid settings together

In [36]:
INVALID_DEPLOYMENT_TOML = r"""
[application]
name = ""
environment = "moon"
replicas = 0

[application.resources]
cpu = -1.0
memory_mb = 64

[features]
audit_log = "yes"
"""

try:
    load_deployment_config(
        INVALID_DEPLOYMENT_TOML,
        source_name="invalid-deployment.toml",
    )
except ExceptionGroup as ex:
    deployment_errors = ex

assert len(deployment_errors.exceptions) == 6
assert deployment_errors.__notes__ == [
    "configuration_source='invalid-deployment.toml'"
]

### Lessons

- `tomllib.load()` expects a binary file object.
- Parsing and schema validation are separate stages.
- Use `parse_float=Decimal` only when exact decimal semantics are useful.
- `tomllib` is intentionally read-only.
- Limit untrusted input size before parsing.

---

## Problem 7 — Model a workflow protocol with `StrEnum`

`StrEnum` members are also strings, which is useful at TOML, JSON, HTTP, CLI, and message-queue boundaries.

### Scenario

A document workflow accepts lowercase state and command strings. Only a small set of transitions is legal.

### Step 1 — Define stable external values

In [37]:
class DocumentState(StrEnum):
    DRAFT = "draft"
    REVIEW = "review"
    APPROVED = "approved"
    ARCHIVED = "archived"


class DocumentCommand(StrEnum):
    SUBMIT = "submit"
    APPROVE = "approve"
    REJECT = "reject"
    ARCHIVE = "archive"

### Step 2 — Store transitions as inspectable data

In [38]:
TRANSITIONS = {
    (DocumentState.DRAFT, DocumentCommand.SUBMIT): DocumentState.REVIEW,
    (DocumentState.REVIEW, DocumentCommand.APPROVE): DocumentState.APPROVED,
    (DocumentState.REVIEW, DocumentCommand.REJECT): DocumentState.DRAFT,
    (DocumentState.APPROVED, DocumentCommand.ARCHIVE): DocumentState.ARCHIVED,
}

### Step 3 — Parse at the boundary and apply the transition

In [39]:
def transition_document(
    current: str | DocumentState,
    command: str | DocumentCommand,
) -> DocumentState:
    try:
        state = DocumentState(current)
    except ValueError as ex:
        ex.add_note(f"unknown document state: {current!r}")
        raise

    try:
        parsed_command = DocumentCommand(command)
    except ValueError as ex:
        ex.add_note(f"unknown document command: {command!r}")
        raise

    try:
        return TRANSITIONS[state, parsed_command]
    except KeyError as ex:
        raise ValueError(
            f"command {parsed_command.value!r} is not legal from {state.value!r}"
        ) from ex

### Step 4 — Test valid and invalid transitions

In [40]:
assert transition_document("draft", "submit") is DocumentState.REVIEW
assert transition_document(DocumentState.REVIEW, "approve") is DocumentState.APPROVED

illegal = assert_raises(ValueError, transition_document, "draft", "archive")
assert "not legal" in str(illegal)

Some legacy APIs use `type(value) is str`. At those boundaries, pass `member.value` or `str(member)` explicitly.

### Lessons

- Convert raw text to enums at the boundary.
- Use explicit values for long-lived protocols.
- Keep simple state machines in data tables.
- Use enum members internally instead of scattered string literals.

---

## Problem 8 — Preserve subclass types in a fluent API with `Self`

`typing.Self`, added in Python 3.11, expresses that a method returns the concrete type of its receiver.

### Scenario

An immutable query builder has a tenant-aware subclass. Chained base-class methods called on the subclass should remain typed as that subclass.

### Step 1 — Implement immutable fluent methods

In [41]:
from dataclasses import replace
from typing import Self


@dataclass(frozen=True, slots=True)
class QueryBuilder:
    table: str
    filters: tuple[str, ...] = ()
    order_columns: tuple[str, ...] = ()
    limit_value: int | None = None

    def where(self, predicate: str) -> Self:
        return replace(self, filters=(*self.filters, predicate))

    def order_by(self, *columns: str) -> Self:
        return replace(self, order_columns=(*self.order_columns, *columns))

    def limit(self, count: int) -> Self:
        if count <= 0:
            raise ValueError("limit must be positive")
        return replace(self, limit_value=count)

    def render(self) -> str:
        text = f"SELECT * FROM {self.table}"
        if self.filters:
            text += " WHERE " + " AND ".join(self.filters)
        if self.order_columns:
            text += " ORDER BY " + ", ".join(self.order_columns)
        if self.limit_value is not None:
            text += f" LIMIT {self.limit_value}"
        return text

### Step 2 — Add the specialized subclass

In [42]:
@dataclass(frozen=True, slots=True)
class TenantQueryBuilder(QueryBuilder):
    tenant_id: int = 0

    def for_current_tenant(self) -> Self:
        return self.where(f"tenant_id = {self.tenant_id}")

### Step 3 — Chain base and subclass methods

In [43]:
query = (
    TenantQueryBuilder(table="invoices", tenant_id=42)
    .for_current_tenant()
    .where("status = 'open'")
    .order_by("due_date")
    .limit(25)
)

assert isinstance(query, TenantQueryBuilder)
assert query.tenant_id == 42
assert query.render() == (
    "SELECT * FROM invoices WHERE tenant_id = 42 AND status = 'open' "
    "ORDER BY due_date LIMIT 25"
)
query.render()

"SELECT * FROM invoices WHERE tenant_id = 42 AND status = 'open' ORDER BY due_date LIMIT 25"

The runtime behavior does not depend on `Self`; the improvement is that a static checker can preserve the concrete subclass across the chain.

### Lessons

- Use `Self` when a return type follows the receiver’s concrete class.
- Do not use it when a method returns a fixed unrelated type.
- Immutability makes fluent builders easier to reuse safely.
- Parameterize security-sensitive values; the example focuses only on typing.

---

## Problem 9 — Describe and validate partial updates

Python 3.11 added `typing.Required` and `typing.NotRequired`, allowing individual `TypedDict` keys to override the class-wide `total` setting.

### Scenario

A user patch requires `user_id`, while editable fields are optional. Unknown fields should be rejected at runtime.

### Step 1 — Describe the static payload shapes

In [44]:
from typing import NotRequired, Required, TypedDict


class CreateUserPayload(TypedDict):
    email: str
    display_name: str
    active: NotRequired[bool]


class UpdateUserPayload(TypedDict, total=False):
    user_id: Required[int]
    email: str
    display_name: str
    active: bool

`TypedDict` does not automatically validate arbitrary dictionaries at runtime, but its required and optional key metadata can support a focused boundary validator.

In [45]:
assert UpdateUserPayload.__required_keys__ == frozenset({"user_id"})
assert UpdateUserPayload.__optional_keys__ == frozenset(
    {"email", "display_name", "active"}
)

### Step 2 — Implement a small runtime validator

In [46]:
def validate_update_payload(payload: dict[str, object]) -> UpdateUserPayload:
    errors: list[Exception] = []
    allowed = UpdateUserPayload.__required_keys__ | UpdateUserPayload.__optional_keys__

    missing = UpdateUserPayload.__required_keys__ - payload.keys()
    unknown = payload.keys() - allowed

    if missing:
        errors.append(ValueError(f"missing required keys: {sorted(missing)!r}"))
    if unknown:
        errors.append(ValueError(f"unknown keys: {sorted(unknown)!r}"))

    expected_types = {
        "user_id": int,
        "email": str,
        "display_name": str,
        "active": bool,
    }

    for key, value in payload.items():
        expected = expected_types.get(key)
        if expected is None:
            continue
        if not isinstance(value, expected) or (expected is int and isinstance(value, bool)):
            errors.append(
                TypeError(
                    f"{key} must be {expected.__name__}, got {type(value).__name__}"
                )
            )

    if errors:
        raise ExceptionGroup("invalid update payload", errors)

    return payload  # type: ignore[return-value]

### Step 3 — Test valid and invalid patches

In [47]:
valid_patch = validate_update_payload(
    {"user_id": 1001, "display_name": "Ada L."}
)
assert valid_patch["user_id"] == 1001

try:
    validate_update_payload({"email": 123, "administrator": True})
except ExceptionGroup as ex:
    patch_errors = exception_leaves(ex)

assert len(patch_errors) == 3
[str(error) for error in patch_errors]

["missing required keys: ['user_id']",
 "unknown keys: ['administrator']",
 'email must be str, got int']

### Lessons

- `TypedDict` is primarily a static shape description.
- Validate untrusted dictionaries at runtime.
- Reject unknown keys when silent typos are dangerous.
- Use a mature schema library for recursive validation and coercion.

---

## Problem 10 — Express a variable-length tensor shape

PEP 646 introduced variadic generics through `TypeVarTuple` and `Unpack`. A type can now be generic over an arbitrary number of type parameters.

### Scenario

A tensor type carries a type-level shape while still validating actual dimensions and flat data length at runtime.

### Step 1 — Define variadic type parameters and the container

In [48]:
from typing import Generic, TypeVar, TypeVarTuple, Unpack


Rows = TypeVar("Rows")
Columns = TypeVar("Columns")
Shape = TypeVarTuple("Shape")
Element = TypeVar("Element")


@dataclass(frozen=True, slots=True)
class Tensor(Generic[Element, Unpack[Shape]]):
    data: tuple[Element, ...]
    shape: tuple[int, ...]

    def __post_init__(self) -> None:
        expected_size = 1
        for dimension in self.shape:
            if dimension <= 0:
                raise ValueError("all dimensions must be positive")
            expected_size *= dimension

        if expected_size != len(self.data):
            raise ValueError(
                f"shape {self.shape!r} requires {expected_size} values, "
                f"got {len(self.data)}"
            )

The type annotation communicates shape relationships to static tools. The constructor enforces the actual size at runtime.

### Step 2 — Implement a typed two-dimensional transpose

In [49]:
def transpose_2d(
    tensor: Tensor[Element, Rows, Columns],
) -> Tensor[Element, Columns, Rows]:
    if len(tensor.shape) != 2:
        raise ValueError("transpose_2d requires exactly two dimensions")

    rows, columns = tensor.shape
    transposed = tuple(
        tensor.data[row * columns + column]
        for column in range(columns)
        for row in range(rows)
    )
    return Tensor(transposed, (columns, rows))

### Step 3 — Verify the transformation and the runtime guard

In [50]:
matrix: Tensor[int, Rows, Columns] = Tensor(
    data=(1, 2, 3, 4, 5, 6),
    shape=(2, 3),
)

transposed = transpose_2d(matrix)
assert transposed.shape == (3, 2)
assert transposed.data == (1, 4, 2, 5, 3, 6)

shape_error = assert_raises(ValueError, Tensor, (1, 2, 3), (2, 2))
assert "requires 4 values" in str(shape_error)

### Lessons

- Use variadic generics only when the number of type parameters is genuinely variable.
- Keep runtime validation even when annotations describe relationships.
- State clearly which guarantees are static and which are runtime-enforced.

---

## Problem 11 — Mark a parameterized query template with `LiteralString`

`typing.LiteralString` lets static tools distinguish literal-derived strings from arbitrary runtime strings.

### Scenario

A repository accepts a fixed SQL template and a separate parameter mapping. The type annotation should discourage callers from interpolating untrusted values into query text.

### Important limitation

`LiteralString` is not enforced at runtime. It complements the database driver’s parameterization mechanism; it does not replace it.

In [51]:
from typing import LiteralString, Protocol


class DatabaseConnection(Protocol):
    def execute(
        self,
        query: str,
        parameters: dict[str, object],
    ) -> list[dict[str, object]]: ...


def execute_template(
    connection: DatabaseConnection,
    query: LiteralString,
    /,
    **parameters: object,
) -> list[dict[str, object]]:
    return connection.execute(query, parameters)

### Step 2 — Test the boundary with a recording fake

In [52]:
@dataclass
class RecordingConnection:
    calls: list[tuple[str, dict[str, object]]]

    def execute(
        self,
        query: str,
        parameters: dict[str, object],
    ) -> list[dict[str, object]]:
        self.calls.append((query, parameters))
        return [{"invoice_id": 7, "status": "open"}]


connection = RecordingConnection(calls=[])
rows = execute_template(
    connection,
    "SELECT invoice_id, status FROM invoices WHERE customer_id = :customer_id",
    customer_id=42,
)

assert connection.calls[0][1] == {"customer_id": 42}
assert rows[0]["invoice_id"] == 7

### Lessons

- Keep values in the parameter mapping.
- Use `LiteralString` to make API intent visible to static tools.
- Depend on the database driver for runtime parameterization.
- Do not write homemade SQL escaping functions.

---

## Problem 12 — Control regular-expression backtracking

Python 3.11 added atomic groups `(?>...)` and possessive quantifiers such as `*+`, `++`, `?+`, and `{m,n}+` to `re`.

### Step 1 — Observe the semantic difference

A greedy quantifier may give characters back. A possessive quantifier consumes them permanently.

In [53]:
import re

assert re.fullmatch(r"a*a", "aaaa") is not None
assert re.fullmatch(r"a*+a", "aaaa") is None

### Scenario

A line protocol accepts `SET key value` and `DELETE key`. Values may be quoted with backslash escapes or may be unquoted tokens.

### Step 2 — Build components with deliberate atomicity

In [54]:
KEY = r"[A-Za-z_][A-Za-z0-9_:-]*+"
QUOTED_VALUE = r'"(?>(?:\\.|[^"\\]))*"'
BARE_VALUE = r'[^\s"]++'
VALUE = rf"(?:{QUOTED_VALUE}|{BARE_VALUE})"

COMMAND_PATTERN = re.compile(
    rf"^(?P<operation>SET|DELETE)\s+"
    rf"(?P<key>{KEY})"
    rf"(?:\s+(?P<value>{VALUE}))?$"
)

The quoted-value atom commits to classifying each character as either an escape or an ordinary non-quote character. The possessive key and bare-value repetitions do not need to give characters back.

### Step 3 — Parse and validate command-specific rules

In [55]:
def decode_quoted_value(value: str) -> str:
    result: list[str] = []
    index = 1
    while index < len(value) - 1:
        character = value[index]
        if character != "\\":
            result.append(character)
            index += 1
            continue

        index += 1
        if index >= len(value) - 1:
            raise ValueError("trailing escape")
        escaped = value[index]
        replacements = {"n": "\n", "t": "\t", "\\": "\\", '"': '"'}
        if escaped not in replacements:
            raise ValueError(f"unsupported escape: \\{escaped}")
        result.append(replacements[escaped])
        index += 1

    return "".join(result)


def parse_command(line: str) -> dict[str, str | None]:
    if len(line) > 1_000:
        raise ValueError("command exceeds maximum length")

    match = COMMAND_PATTERN.fullmatch(line)
    if match is None:
        raise ValueError(f"invalid command: {line!r}")

    operation = match.group("operation")
    value = match.group("value")

    if operation == "SET" and value is None:
        raise ValueError("SET requires a value")
    if operation == "DELETE" and value is not None:
        raise ValueError("DELETE does not accept a value")

    if value is not None and value.startswith('"'):
        value = decode_quoted_value(value)

    return {"operation": operation, "key": match.group("key"), "value": value}

### Step 4 — Test valid and invalid protocol lines

In [56]:
assert parse_command('SET customer:42 "premium member"') == {
    "operation": "SET",
    "key": "customer:42",
    "value": "premium member",
}
assert parse_command("DELETE customer:42")["value"] is None
assert parse_command(r'SET message "line\nnext"')["value"] == "line\nnext"

assert_raises(ValueError, parse_command, "SET customer:42")
assert_raises(ValueError, parse_command, "DELETE customer:42 now")
assert_raises(ValueError, parse_command, 'SET customer:42 "unterminated')

ValueError('invalid command: \'SET customer:42 "unterminated\'')

### Lessons

- Atomicity can improve behavior only when backtracking is unnecessary.
- These constructs change matching semantics; do not add them mechanically.
- Use `fullmatch()` for complete records.
- Apply cheap input-size limits before matching untrusted text.
- Use a real parser when the grammar grows.

---

## Problem 13 — Enforce an application limit before decimal conversion

Python 3.11 introduced a configurable limit for decimal string-to-integer conversions, reducing the risk of excessive CPU consumption from enormous inputs.

### Scenario

Account identifiers are decimal strings of at most 64 ASCII digits. Signs, whitespace, separators, and non-ASCII numerals are forbidden.

### Step 1 — Inspect the interpreter-wide limit

In [57]:
current_digit_limit = sys.get_int_max_str_digits()
current_digit_limit

4300

A zero value means the interpreter-wide limit is disabled. Domain code should still enforce its own smaller limit before calling `int()`.

### Step 2 — Validate lexical form and length

In [58]:
def parse_account_id(text: str, *, max_digits: int = 64) -> int:
    if not isinstance(text, str):
        raise TypeError("account ID must be supplied as text")
    if not text:
        raise ValueError("account ID cannot be empty")
    if len(text) > max_digits:
        raise ValueError(f"account ID exceeds the {max_digits}-digit application limit")
    if any(character < "0" or character > "9" for character in text):
        raise ValueError("account ID must contain ASCII digits only")
    return int(text)

### Step 3 — Test the domain rules and the interpreter guard

In [59]:
assert parse_account_id("000123") == 123
assert_raises(ValueError, parse_account_id, "")
assert_raises(ValueError, parse_account_id, "+123")
assert_raises(ValueError, parse_account_id, "１２３")
assert_raises(ValueError, parse_account_id, "9" * 65)

if current_digit_limit:
    enormous_decimal = "9" * (current_digit_limit + 1)
    conversion_error = assert_raises(ValueError, int, enormous_decimal)
    assert "limit" in str(conversion_error).lower()

### Lessons

- Apply a small domain limit before conversion.
- Validate ASCII explicitly when the protocol requires it.
- Avoid changing global interpreter settings inside a library.
- Keep identifiers as strings when arithmetic is unnecessary or leading zeros matter.

---

## Problem 14 — Capstone: resilient concurrent report jobs

This final problem combines `tomllib`, `StrEnum`, `TaskGroup`, `asyncio.timeout()`, exception notes, `ExceptionGroup`, `TypedDict`, and structural pattern matching.

### Scenario

A TOML file defines independent numeric report jobs. All jobs should run concurrently. Successful results should be preserved, while every expected job failure is reported together after all jobs finish.

### Step 1 — Define the configuration

In [60]:
REPORT_TOML = r"""
[[jobs]]
name = "daily-total"
kind = "sum"
values = [10, 20, 30]
timeout_ms = 100

[[jobs]]
name = "normalized"
kind = "normalize"
values = [3, 6, 9]
timeout_ms = 100

[[jobs]]
name = "empty-average"
kind = "average"
values = []
timeout_ms = 100

[[jobs]]
name = "slow-total"
kind = "sum"
values = [1, 2, 3]
timeout_ms = 5
simulate_delay_ms = 30
"""

### Step 2 — Define the external values and domain records

In [61]:
class JobKind(StrEnum):
    SUM = "sum"
    AVERAGE = "average"
    NORMALIZE = "normalize"


@dataclass(frozen=True, slots=True)
class ReportJob:
    name: str
    kind: JobKind
    values: tuple[float, ...]
    timeout_seconds: float
    simulate_delay_seconds: float = 0.0


class SuccessfulJob(TypedDict):
    name: str
    kind: str
    value: float | list[float]

### Step 3 — Parse every job and group configuration errors

In [62]:
def report_jobs_from_toml(text: str) -> list[ReportJob]:
    document = tomllib.loads(text)
    raw_jobs = document.get("jobs")
    if not isinstance(raw_jobs, list):
        raise ValueError("configuration must contain [[jobs]] tables")

    jobs: list[ReportJob] = []
    errors: list[Exception] = []

    for index, raw_job in enumerate(raw_jobs):
        try:
            if not isinstance(raw_job, dict):
                raise TypeError("job must be a table")

            name = raw_job["name"]
            kind = JobKind(raw_job["kind"])
            values = raw_job["values"]
            timeout_ms = raw_job["timeout_ms"]
            delay_ms = raw_job.get("simulate_delay_ms", 0)

            if not isinstance(name, str) or not name:
                raise ValueError("job name must be a non-empty string")
            if not isinstance(values, list) or any(
                not isinstance(value, (int, float)) or isinstance(value, bool)
                for value in values
            ):
                raise TypeError("job values must be a list of numbers")
            if not isinstance(timeout_ms, int) or timeout_ms <= 0:
                raise ValueError("timeout_ms must be a positive integer")
            if not isinstance(delay_ms, int) or delay_ms < 0:
                raise ValueError("simulate_delay_ms must be non-negative")

            jobs.append(
                ReportJob(
                    name=name,
                    kind=kind,
                    values=tuple(float(value) for value in values),
                    timeout_seconds=timeout_ms / 1000,
                    simulate_delay_seconds=delay_ms / 1000,
                )
            )
        except (KeyError, TypeError, ValueError) as ex:
            ex.add_note(f"job_index={index}")
            errors.append(ex)

    if errors:
        raise ExceptionGroup("invalid report job configuration", errors)
    return jobs

### Step 4 — Keep numeric transformations pure

In [63]:
def compute_report_value(job: ReportJob) -> float | list[float]:
    match job.kind:
        case JobKind.SUM:
            return sum(job.values)
        case JobKind.AVERAGE:
            if not job.values:
                raise ValueError("average requires at least one value")
            return sum(job.values) / len(job.values)
        case JobKind.NORMALIZE:
            if not job.values:
                raise ValueError("normalize requires at least one value")
            largest = max(abs(value) for value in job.values)
            if largest == 0:
                raise ValueError("cannot normalize an all-zero sequence")
            return [value / largest for value in job.values]

### Step 5 — Convert expected job failures into attempt values

This lets independent jobs continue. Unexpected orchestration failures can still escape and trigger fail-fast `TaskGroup` behavior.

In [64]:
@dataclass(frozen=True, slots=True)
class JobAttempt:
    success: SuccessfulJob | None
    error: Exception | None


async def execute_report_job(job: ReportJob) -> JobAttempt:
    try:
        async with asyncio.timeout(job.timeout_seconds):
            await asyncio.sleep(job.simulate_delay_seconds)
            value = compute_report_value(job)
    except Exception as ex:
        ex.add_note(f"job_name={job.name!r}")
        ex.add_note(f"job_kind={job.kind.value!r}")
        return JobAttempt(None, ex)

    return JobAttempt(
        SuccessfulJob(name=job.name, kind=job.kind.value, value=value),
        None,
    )

### Step 6 — Run all jobs, then aggregate the failures

In [65]:
async def run_report_jobs(jobs: list[ReportJob]) -> list[SuccessfulJob]:
    tasks: list[asyncio.Task[JobAttempt]] = []

    async with asyncio.TaskGroup() as group:
        for job in jobs:
            tasks.append(
                group.create_task(
                    execute_report_job(job),
                    name=f"report:{job.name}",
                )
            )

    attempts = [task.result() for task in tasks]
    successes = [attempt.success for attempt in attempts if attempt.success is not None]
    errors = [attempt.error for attempt in attempts if attempt.error is not None]

    if errors:
        group = ExceptionGroup("one or more report jobs failed", errors)
        group.add_note(f"successful_jobs={len(successes)}")
        raise group

    return successes

### Step 7 — Select timeout and value failures separately

In [66]:
report_jobs = report_jobs_from_toml(REPORT_TOML)
report_value_errors: list[Exception] = []
report_timeouts: list[Exception] = []

try:
    await run_report_jobs(report_jobs)
except* TimeoutError as group:
    report_timeouts.extend(exception_leaves(group))
except* ValueError as group:
    report_value_errors.extend(exception_leaves(group))

assert len(report_timeouts) == 1
assert len(report_value_errors) == 1
assert report_timeouts[0].__notes__ == [
    "job_name='slow-total'", "job_kind='sum'"
]
assert report_value_errors[0].__notes__ == [
    "job_name='empty-average'", "job_kind='average'"
]

### Step 8 — Verify a fully successful run

In [67]:
success_only = [
    job for job in report_jobs if job.name in {"daily-total", "normalized"}
]
results = await run_report_jobs(success_only)

assert results == [
    {"name": "daily-total", "kind": "sum", "value": 60.0},
    {
        "name": "normalized",
        "kind": "normalize",
        "value": [1 / 3, 2 / 3, 1.0],
    },
]
results

[{'name': 'daily-total', 'kind': 'sum', 'value': 60.0},
 {'name': 'normalized',
  'kind': 'normalize',
  'value': [0.3333333333333333, 0.6666666666666666, 1.0]}]

### Step 9 — Review the boundaries

1. TOML parsing converts text to generic containers.  
2. Schema validation converts containers to domain objects.  
3. Pure functions perform deterministic calculations.  
4. The asynchronous layer owns concurrency and deadlines.  
5. Notes attach job identity.  
6. The final exception group aggregates independent failures.

### Capstone lessons

- Expected child failures can be modeled as values to preserve partial progress.
- Aggregate only after independent work finishes.
- Keep computation separate from orchestration.
- Attach context close to the failing job.
- Use task names, local timeouts, and domain-specific models together.

---

## Extra worked examples

### Example A — `StrEnum.auto()` produces lower-case values

In [68]:
from enum import auto


class LogLevel(StrEnum):
    DEBUG = auto()
    INFO = auto()
    WARNING = auto()
    ERROR = auto()


assert LogLevel.DEBUG.value == "debug"
assert LogLevel.WARNING.value == "warning"

Use explicit values instead when an external protocol must remain stable even if member names change.

### Example B — Combine exception chaining and a note

In [69]:
class ConfigurationLookupError(RuntimeError):
    pass


def require_setting(settings: dict[str, str], key: str) -> str:
    try:
        return settings[key]
    except KeyError as ex:
        translated = ConfigurationLookupError(f"required setting {key!r} is missing")
        translated.add_note("check the active environment-specific configuration")
        raise translated from ex


translated_error = assert_raises(
    ConfigurationLookupError,
    require_setting,
    {},
    "DATABASE_URL",
)
assert isinstance(translated_error.__cause__, KeyError)
assert translated_error.__notes__ == [
    "check the active environment-specific configuration"
]

Chaining preserves the technical cause; the note adds an operational suggestion.

### Example C — Select an exception subgroup by predicate

In [70]:
mixed_group = ExceptionGroup(
    "mixed failures",
    [
        ValueError("bad value"),
        RuntimeError("temporary outage"),
        ExceptionGroup(
            "nested",
            [ValueError("another bad value"), TypeError("wrong type")],
        ),
    ],
)

value_only = mixed_group.subgroup(lambda ex: isinstance(ex, ValueError))
assert value_only is not None
assert [str(ex) for ex in exception_leaves(value_only)] == [
    "bad value", "another bad value"
]

### Example D — Reschedule a timeout after entering the context

In [71]:
async def operation_with_late_budget(work_time: float, budget: float) -> str:
    loop = asyncio.get_running_loop()
    async with asyncio.timeout(None) as timeout_context:
        timeout_context.reschedule(loop.time() + budget)
        await asyncio.sleep(work_time)
        return "completed"


assert await operation_with_late_budget(0.005, 0.05) == "completed"

## Review checklist

You should now be able to explain:

- how fine-grained source positions improve diagnostics;
- when notes are preferable to replacing an exception;
- how nested exception groups preserve batch structure;
- why `except*` differs from ordinary `except`;
- how `TaskGroup` defines child-task lifetime;
- when to use fail-fast versus tolerant concurrency;
- how local and global timeout budgets interact;
- why TOML parsing and schema validation are separate;
- what `StrEnum`, `Self`, `Required`, `NotRequired`, `TypeVarTuple`, `Unpack`, and `LiteralString` contribute;
- how possessive quantifiers and atomic groups change backtracking;
- why domain-specific integer limits should be checked before conversion.

## Further advanced exercises with solution directions

### Exercise 1 — Retry only timeout failures

Run failed timeout jobs once more without rerunning value failures.

**Solution direction:** retain each `ReportJob` in `JobAttempt`, select timeout attempts, run a second `TaskGroup`, and add a `retry=1` note to failures that remain.

### Exercise 2 — Nested feature configuration

Allow each feature to be either a Boolean or a table containing `enabled` and `rollout_percentage`.

**Solution direction:** use structural pattern matching to parse each value, raise one nested group per invalid feature, and preserve the feature name in a note.

### Exercise 3 — A custom exception-group subclass

Create `ReportFailureGroup` with a `successes` attribute.

**Solution direction:** subclass `ExceptionGroup` and override `derive()` so subgroup operations preserve custom metadata. Follow the subclassing rules in PEP 654.

### Exercise 4 — Three-dimensional axis swapping

Implement `swap_first_two_axes(Tensor[Element, X, Y, Z]) -> Tensor[Element, Y, X, Z]`.

**Solution direction:** derive source and destination flat indexes explicitly and retain runtime shape validation.

### Exercise 5 — Harden the command parser

Reject control characters and support only an explicit escape allow-list.

**Solution direction:** perform cheap lexical checks before regex matching and keep escape decoding in a separate deterministic function.

---

## Closing perspective

The strongest Python 3.11 designs attach features to clear boundaries: positions at diagnostic boundaries, notes at context boundaries, groups at aggregation boundaries, task groups at concurrency boundaries, timeouts at latency boundaries, and typing at interface boundaries.

That approach produces code that is easier to debug, safer to evolve, and more explicit about failure and concurrency semantics.